# Evaluación orientada a negocio

Calcula indicadores de tráfico a partir de las fases del preprocesamiento unidas por `(camera, image)`:
`frames` (contexto temporal e iluminación) + `weather` + `detections` (conteos por clase y estado de semáforos).

Bloques de indicadores:

**1. Tráfico**
- Vehículos y peatones medios por imagen (ocupación instantánea; con imágenes fijas no se puede medir un flujo por minuto sin seguir objetos entre fotogramas).
- Fracción del tiempo en rojo / ámbar / verde del semáforo vehicular.
- Duración media de las fases verde y roja (**solo si el refresco medido lo permite**, ver abajo).

**2. Comportamiento**
- Peatones esperando con semáforo peatonal en rojo / cruzando en verde.
- Vehículos acumulados con semáforo vehicular en rojo frente a verde (cola estimada).

**3. Eficiencia**
- Tiempo medio de espera (si se pueden medir duraciones).
- Densidad vehicular y ratio peatón/vehículo.

**Duraciones de fase.** Un ciclo semafórico dura ~60–120 s. Las duraciones solo se calculan si el refresco medido en `00b` es ≤ `max_refresco_para_duraciones_s` y las rachas se cortan cuando el hueco entre dos imágenes consecutivas supera `max_hueco_racha_s` (config.yaml → `evaluacion`). Si no se cumple, se informa la **fracción de tiempo** en cada estado, que sí es estimable con muestras de fase aleatoria.

Los indicadores se calculan por cámara, por franja horaria y por contexto (iluminación y clima).

## 1. Carga de datos y configuración

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from config import load_config, path
from deteccion import VEHICLE_CLASSES
from pipeline import load_frames, measured_refresh_s

CFG = load_config()["evaluacion"]
RESULTS_DIR = path("resultados")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

frames = load_frames("weather", "detections")
if "n_vehiculos" not in frames.columns:
    raise RuntimeError("Falta la fase detections: ejecuta antes 02_preprocess_labelled_detection.ipynb")
frames = frames.dropna(subset=["n_vehiculos"]).sort_values(["camera", "timestamp"]).reset_index(drop=True)

REFRESH_S = measured_refresh_s()
DURATIONS_OK = REFRESH_S is not None and REFRESH_S <= CFG["max_refresco_para_duraciones_s"]
MAX_GAP_S = CFG["max_hueco_racha_s"]

print(f"Fotogramas: {len(frames)} | Cámaras: {frames['camera'].nunique()}")
print(f"Refresco medido: {REFRESH_S if REFRESH_S is not None else 'sin medición (ejecuta 00b)'} s")
print(f"Duraciones de fase: {'se calculan' if DURATIONS_OK else 'NO (se usan fracciones de tiempo)'}")

## 2. Estado del semáforo por fotograma

Con varios semáforos en la imagen, el estado es el color común a todos; si hay colores distintos el fotograma se marca como `mixto` y no se usa para los indicadores de estado.

In [ ]:
def light_state(row, prefix, colors):
    present = [c for c in colors if row.get(f"n_{prefix}_{c}", 0) > 0]
    if not present:
        return None
    return present[0] if len(present) == 1 else "mixto"


frames["vehicle_light"] = frames.apply(lambda r: light_state(r, "traffic_light", ["red", "yellow", "green"]), axis=1)
frames["pedestrian_light"] = frames.apply(lambda r: light_state(r, "pedestrian_light", ["red", "green"]), axis=1)

print("Estado semáforo vehicular:")
print(frames["vehicle_light"].value_counts(dropna=False))
print("\nEstado semáforo peatonal:")
print(frames["pedestrian_light"].value_counts(dropna=False))

In [ ]:
def run_durations(g, column, state, max_gap_s):
    """Duraciones (s) de las rachas completas de `state` usando los timestamps reales.

    Una racha empieza en la primera imagen con ese estado y termina en la primera con otro
    estado. Se descartan las rachas cortadas por un hueco > max_gap_s y las censuradas
    (al principio o al final de la serie), porque su duración real es desconocida.
    """
    durations = []
    start = None
    prev_ts = None
    prev_state = None
    for ts, s in zip(g["timestamp"], g[column]):
        if prev_ts is not None and (ts - prev_ts).total_seconds() > max_gap_s:
            # Hueco: la racha en curso queda incompleta y la transición siguiente no se ha observado
            start = None
            prev_state = None
        elif start is not None and s != state:
            durations.append((ts - start).total_seconds())
            start = None
        if s == state and start is None and prev_state is not None and prev_state != state:
            start = ts  # solo rachas cuyo inicio se ha observado (transición desde otro estado conocido)
        prev_ts, prev_state = ts, s
    return durations


def state_fractions(g, column, colors):
    known = g[column].isin(colors)
    if not known.any():
        return {f"frac_{c}": np.nan for c in colors}
    return {f"frac_{c}": float((g.loc[known, column] == c).mean()) for c in colors}

## 3. Indicadores por cámara

In [ ]:
def camera_indicators(g):
    red = g["vehicle_light"] == "red"
    green = g["vehicle_light"] == "green"
    ped_red = g["pedestrian_light"] == "red"
    ped_green = g["pedestrian_light"] == "green"

    row = {
        "n_frames": len(g),
        # Tráfico
        "vehiculos_medios_por_imagen": g["n_vehiculos"].mean(),
        "vehiculos_p90_por_imagen": g["n_vehiculos"].quantile(0.9),
        "peatones_medios_por_imagen": g["n_person"].mean(),
        **state_fractions(g, "vehicle_light", ["red", "yellow", "green"]),
        # Comportamiento
        "peatones_esperando_rojo": g.loc[ped_red, "n_person"].mean() if ped_red.any() else np.nan,
        "peatones_cruzando_verde": g.loc[ped_green, "n_person"].mean() if ped_green.any() else np.nan,
        "vehiculos_acumulados_rojo": g.loc[red, "n_vehiculos"].mean() if red.any() else np.nan,
        "vehiculos_en_verde": g.loc[green, "n_vehiculos"].mean() if green.any() else np.nan,
        # Eficiencia
        "densidad_vehicular": g["n_vehiculos"].mean(),
        "ratio_peaton_vehiculo": g["n_person"].sum() / g["n_vehiculos"].sum() if g["n_vehiculos"].sum() else np.nan,
    }

    if DURATIONS_OK:
        verdes = run_durations(g, "vehicle_light", "green", MAX_GAP_S)
        rojas = run_durations(g, "vehicle_light", "red", MAX_GAP_S)
        espera = run_durations(g.assign(_espera=np.where(red & (g["n_person"] > 0), "espera", "otro")), "_espera", "espera", MAX_GAP_S)
        row.update({
            "duracion_verde_media_s": np.mean(verdes) if verdes else np.nan,
            "duracion_roja_media_s": np.mean(rojas) if rojas else np.nan,
            "n_fases_medidas": len(verdes) + len(rojas),
            "tiempo_medio_espera_s": np.mean(espera) if espera else np.nan,
        })
    return pd.Series(row)


camera_df = frames.groupby("camera").apply(camera_indicators, include_groups=False).reset_index()
display(camera_df)

print("=== Agregado global (media por cámara) ===")
display(camera_df.drop(columns=["camera"]).mean().to_frame("media"))

## 4. Indicadores por franja horaria y por contexto

Los mismos indicadores (sin duraciones, que requieren series continuas por cámara) desglosados por franja horaria, tipo de día, iluminación y clima.

In [ ]:
def static_indicators(g):
    red = g["vehicle_light"] == "red"
    return pd.Series({
        "n_frames": len(g),
        "n_camaras": g["camera"].nunique(),
        "vehiculos_medios_por_imagen": g["n_vehiculos"].mean(),
        "peatones_medios_por_imagen": g["n_person"].mean(),
        **state_fractions(g, "vehicle_light", ["red", "yellow", "green"]),
        "vehiculos_acumulados_rojo": g.loc[red, "n_vehiculos"].mean() if red.any() else np.nan,
        "ratio_peaton_vehiculo": g["n_person"].sum() / g["n_vehiculos"].sum() if g["n_vehiculos"].sum() else np.nan,
    })


slot_df = frames.groupby("time_slot_2h").apply(static_indicators, include_groups=False).reset_index()
display(slot_df)

context_cols = [c for c in ["day_type", "illumination", "weather"] if c in frames.columns]
context_df = pd.concat(
    [frames.groupby(col).apply(static_indicators, include_groups=False).reset_index().rename(columns={col: "valor"}).assign(contexto=col)
     for col in context_cols],
    ignore_index=True,
)
display(context_df.set_index(["contexto", "valor"]))

## 5. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(slot_df["time_slot_2h"], slot_df["vehiculos_medios_por_imagen"], marker="o", label="Vehículos/imagen")
axes[0, 0].plot(slot_df["time_slot_2h"], slot_df["peatones_medios_por_imagen"], marker="s", label="Peatones/imagen")
axes[0, 0].set_title("Ocupación media por franja horaria")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)
axes[0, 0].tick_params(axis="x", rotation=45)

fractions = camera_df.set_index("camera")[["frac_red", "frac_yellow", "frac_green"]].dropna(how="all")
if not fractions.empty:
    fractions.plot(kind="bar", stacked=True, color=["#d62728", "#ffbf00", "#2ca02c"], ax=axes[0, 1], legend=True)
axes[0, 1].set_title("Fracción del tiempo por estado del semáforo vehicular")
axes[0, 1].set_ylabel("fracción de imágenes")

axes[1, 0].bar(camera_df["camera"], camera_df["densidad_vehicular"])
axes[1, 0].set_title("Densidad vehicular media por cámara")
axes[1, 0].tick_params(axis="x", rotation=90)

axes[1, 1].bar(camera_df["camera"], camera_df["ratio_peaton_vehiculo"])
axes[1, 1].set_title("Ratio peatón/vehículo por cámara")
axes[1, 1].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

## 6. Exportar indicadores consolidados

In [ ]:
camera_df.to_csv(RESULTS_DIR / "indicadores_por_camara.csv", index=False)
slot_df.to_csv(RESULTS_DIR / "indicadores_por_franja.csv", index=False)
context_df.to_csv(RESULTS_DIR / "indicadores_por_contexto.csv", index=False)
print(f"Indicadores guardados en {RESULTS_DIR}")